imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

paths

In [0]:
src_bronze_sales="/Volumes/retail_catalog/retail_bronze/bronze_sales_volume"
target_silver_sales="/Volumes/retail_catalog/retail_silver/silver_sales_volume"
quarantine_sales_path="/Volumes/retail_catalog/retail_quarantine/q_sales_volume"
audit_path="/Volumes/retail_catalog/audit/audit_volume/etl_audit/"

load bronze df

In [0]:

df_sales_bronze=spark.read\
    .format("delta")\
    .load(src_bronze_sales)

incremental df

In [0]:

if DeltaTable.isDeltaTable(spark, target_silver_sales):
    bronze_table = DeltaTable.forPath(spark, target_silver_sales)
    # Get max data_arrival_timestamp
    max_ts_row = bronze_table.toDF().select(max("data_arrival_timestamp")).collect()[0]
    max_ts = max_ts_row[0]  # None if table is empty
    if max_ts is None:
        print("Bronze table is empty. Will load all records.")
else:
    print("silver table not found. Will load all records.(initial load) ")
    max_ts = None  # first load

# Filter source for incremental load
if max_ts:
    df = df_sales_bronze.filter(col("data_arrival_timestamp") > max_ts)
else:
    df = df_sales_bronze  # first load, take all records

print(f"Number of records to load: {df.count()}")

In [0]:
if df.isEmpty():
    dbutils.notebook.exit("no new data to load into silver sales")

filter data based on schema evolution

In [0]:

rescue_df = df.filter(
    col("_rescued_data").isNotNull() &
    (trim(col("_rescued_data")) != "")
)
df=df.filter(col("_rescued_data").isNull())

quarantine schema changed data

In [0]:
if not rescue_df.isEmpty():
    print("data found in rescued column")
    rescue_df = (
    rescue_df.withColumn("layer", lit("silver"))\
             .withColumn("quarantine_timestamp", current_timestamp())
    )
    rescue_df.write\
        .format("delta")\
        .mode("append")\
        .save(quarantine_sales_path)
else:
    pass

In [0]:
if df.isEmpty():
    dbutils.notebook.exit("no new data to load into silver sales")

drop duplicate

In [0]:
df=df.dropDuplicates(['order_id','data_arrival_timestamp'])

drop nulls on imp columns

In [0]:
df = df.dropna(subset=["order_id", "order_date","ship_date","customer_id","product_id","sales","data_arrival_timestamp"],how='any')

handle na 

In [0]:
df=df.fillna({
    "quantity":1,
    "discount":0,
    "profit":0
})

invalid checks and custom business logics

In [0]:
df = df.filter(col("order_id").startswith("ORDER"))

df = df.filter(col("ship_date") >= col("order_date"))

df = df.filter(col("order_date") <= current_date())

df = df.filter(col("data_arrival_timestamp") > col("order_date"))

df=df.filter((col("profit") <= col("sales")))

soft delete flag column

In [0]:
df = df.withColumn("is_deleted", lit(False))

cast data type for all columns

In [0]:
df = (
    df.withColumn("order_id", col("order_id").cast(StringType()))\
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))\
    .withColumn("ship_date", to_date(col("ship_date"), "yyyy-MM-dd"))\
    .withColumn("ship_mode", col("ship_mode").cast(StringType()))\
    .withColumn("customer_id", col("customer_id").cast(StringType()))\
    .withColumn("product_id", col("product_id").cast(StringType()))\
    .withColumn("sales", col("sales").cast(DoubleType()))\
    .withColumn("quantity", col("quantity").cast(IntegerType()))\
    .withColumn("discount", col("discount").cast(DoubleType()))\
    .withColumn("profit", col("profit").cast(DoubleType()))\
    .withColumn("data_arrival_timestamp", to_timestamp(col("data_arrival_timestamp")))\
    .withColumn("is_deleted", lit(False).cast(BooleanType()))
)

In [0]:
df = (
    df.withColumn("order_year", year("order_date"))
    .withColumn("order_month", month("order_date"))
)

In [0]:
df = df.drop("_rescued_data")

In [0]:
df.write\
    .format("delta")\
    .mode("append")\
    .partitionBy("order_year", "order_month")\
    .save("/Volumes/retail_catalog/retail_silver/silver_sales_volume")

add meta data and etl status to audit table

In [0]:

# Count records
records_count = df.count()

if records_count==0:
    msg=" (Source Empty)"
else:
    msg=""

# max timestamp (only if rows exist)
max_data_ts_row = (
    df.select(max("data_arrival_timestamp")).collect()[0][0]
    if records_count > 0
    else None
)

# Use Python datetime for load_time
load_time = datetime.now()

# Define schema explicitly
schema = StructType([
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_time", TimestampType(), True),
    StructField("records_loaded", LongType(), True),
    StructField("max_data_timestamp", TimestampType(), True)
])

# Prepare audit data (even if 0 rows)
data = [("silver", f"silver_customer{msg}", load_time, records_count, max_data_ts_row)]

# Create DataFrame
df_audit = spark.createDataFrame(data, schema)

# Append to audit table
df_audit.write.format("delta") \
    .mode("append") \
    .save(audit_path)

print(f"Audit log updated successfully. Records loaded: {records_count}")
